# 詩織 LoRA v4 顔切り取り仕上げ推論ノートブック（Kaggle）

学習済みLoRA（train-shiori-lora-kaggle-v4 Version 2 の出力 shiori_oneesan_v4.safetensors）を **Add Input** で接続して使う。学習は行わない。
v4b（1.5倍拡大＋全体img2img）は無料GPUのメモリ不足（CUDA OOM）で失敗したため、今回は仕上げを **顔だけ** に絞る：
- ベース生成は 1024px・40ステップ（v4bと同じ条件、シード42）
- アニメ顔検出（Bingsu/adetailer の face_yolov8n）で顔の範囲を自動検出し、余白付きで切り取る
- 切り取った顔だけを 1024px に拡大して img2img（strength 0.4）で描き直し、境界をぼかして貼り戻す
- 顔が検出できなかった場合は、画像上部中央の固定範囲を顔として扱うフォールバック付き
生成は2枚：①標準プロンプト（商店街カフェ）、②髪チェック用（肩に流した髪のアップ）。

In [ ]:
# GPU確認
!nvidia-smi


In [ ]:
# 依存関係の用意（torchao は古い版が入っていると peft が ImportError になるため外す）
!pip install -q diffusers transformers accelerate peft huggingface_hub ultralytics
!pip uninstall -y torchao

# ベースモデル（Animagine XL 4.0）と顔検出モデルを取得
from huggingface_hub import hf_hub_download
import shutil, os
p = hf_hub_download(repo_id='cagliostrolab/animagine-xl-4.0', filename='animagine-xl-4.0.safetensors')
shutil.copy(p, '/kaggle/working/animagine-xl-4.0.safetensors')
print('model ready:', os.path.getsize('/kaggle/working/animagine-xl-4.0.safetensors'))
f = hf_hub_download(repo_id='Bingsu/adetailer', filename='face_yolov8n.pt')
shutil.copy(f, '/kaggle/working/face_yolov8n.pt')
print('face detector ready:', os.path.getsize('/kaggle/working/face_yolov8n.pt'))


In [ ]:
# ベース生成（1024px・40ステップ）→ 顔だけ切り取って仕上げ → 貼り戻し
import glob, os, gc, torch
from diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline
from PIL import Image, ImageDraw, ImageFilter
from ultralytics import YOLO
cands = [c for c in glob.glob('/kaggle/input/**/shiori_oneesan_v4.safetensors', recursive=True)]
print('lora candidates:', cands)
assert cands, 'shiori_oneesan_v4.safetensors not found in /kaggle/input'
lora_path = cands[0]
lora_dir = os.path.dirname(lora_path)
print('lora size:', os.path.getsize(lora_path))
neg = 'lowres, bad anatomy, bad hands, extra fingers, worst quality, blurry, multiple girls, 2girls, extra person, tsurime, sharp eyes, narrow eyes'
os.makedirs('/kaggle/working/output', exist_ok=True)
prompts = [
    ('test_outing_v4c.png', 'shiori_oneesan, 1woman, solo, very long hair, straight black hair, matte black hair, hime cut, blunt bangs, side part, purple eyes, gentle eyes, droopy eyes, tareme, gentle smile, soft expression, pink cardigan, shopping street, cafe terrace, sunny day'),
    ('test_hair_v4c.png', 'shiori_oneesan, 1woman, solo, very long hair, straight black hair, matte black hair, hime cut, blunt bangs, side part, purple eyes, gentle eyes, droopy eyes, tareme, gentle smile, soft expression, hair over shoulder, white blouse, indoors, window, daylight, close-up portrait'),
]

pipe = StableDiffusionXLPipeline.from_single_file('/kaggle/working/animagine-xl-4.0.safetensors', torch_dtype=torch.float16).to('cuda')
pipe.load_lora_weights(lora_dir, weight_name='shiori_oneesan_v4.safetensors')
bases = []
for name, prompt in prompts:
    g = torch.Generator(device='cuda').manual_seed(42)
    img = pipe(prompt, negative_prompt=neg, width=1024, height=1024, num_inference_steps=40, guidance_scale=7, generator=g).images[0]
    bases.append((name, prompt, img))
    print('base done:', name)

# img2img は同じ部品を共有して作る（追加のメモリ消費なし）
pipe2 = StableDiffusionXLImg2ImgPipeline(**pipe.components)
detector = YOLO('/kaggle/working/face_yolov8n.pt')

def find_face_box(img):
    res = detector.predict(img, conf=0.35, verbose=False)
    boxes = []
    for r in res:
        for b in r.boxes:
            x1, y1, x2, y2 = [float(v) for v in b.xyxy[0].tolist()]
            boxes.append((x2 - x1) * (y2 - y1), x1, y1, x2, y2)
    if not boxes:
        return None
    boxes.sort(reverse=True)
    _, x1, y1, x2, y2 = boxes[0]
    w, h = img.size
    mx = (x2 - x1) * 0.45
    my = (y2 - y1) * 0.45
    x1 = max(0, int(x1 - mx)); y1 = max(0, int(y1 - my))
    x2 = min(w, int(x2 + mx)); y2 = min(h, int(y2 + my))
    side = max(x2 - x1, y2 - y1)
    cx = (x1 + x2) // 2; cy = (y1 + y2) // 2
    x1 = max(0, min(w - side, cx - side // 2)); y1 = max(0, min(h - side, cy - side // 2))
    return (x1, y1, x1 + side, y1 + side)

for name, prompt, img in bases:
    box = find_face_box(img)
    if box is None:
        box = (212, 40, 812, 640)
        print('face not detected, fallback box:', name, box)
    else:
        print('face box:', name, box)
    crop = img.crop(box).resize((1024, 1024), Image.LANCZOS)
    g = torch.Generator(device='cuda').manual_seed(42)
    refined = pipe2(prompt, negative_prompt=neg, image=crop, strength=0.4, num_inference_steps=30, guidance_scale=7, generator=g).images[0]
    refined = refined.resize((box[2] - box[0], box[3] - box[1]), Image.LANCZOS)
    mask = Image.new('L', refined.size, 255)
    md = ImageDraw.Draw(mask)
    inset = max(6, int(refined.size[0] * 0.04))
    md.rectangle([0, 0, refined.size[0], refined.size[1]], fill=0)
    md.rectangle([inset, inset, refined.size[0] - inset, refined.size[1] - inset], fill=255)
    mask = mask.filter(ImageFilter.GaussianBlur(radius=inset))
    out = img.copy()
    out.paste(refined, (box[0], box[1]), mask)
    out.save('/kaggle/working/output/' + name)
    print('saved', name)
bases[0][2]
